# LHRM — Independent AAPD TEST-ONLY Evaluation (Frozen R_AUTHOR, Epoch 6)

**Purpose:** Independently evaluate the **previously trained** R_AUTHOR/BCE neural model on the AAPD test set; **NO TRAINING**, no optimizer, no checkpoint selection, no threshold tuning and **NO HRRM fitting**.

**Frozen before examining Test:**
- Run: `AAPD_R_AUTHOR_BCE_RECOVERY_20261015_A`
- Selected model: `best_dev_model_only.pt`, Epoch **6**, Step **40380**, selected by Dev(400) P@1 **0.8950**.
- Label order: exact 54 raw AAPD column names with SHA-256 `a3d662f3920259a94ed61d83f98f21cabe7f908d98a5662ccd9adc2413cc000b`.
- Text model: pinned `google-bert/bert-base-uncased` commit `86b5e0934494bd15c9632b12f734a8a67f723594` (config + tokenizer files needed; the best checkpoint contains **trained BERT model weights**).
- Inference: `max_len=512`, dynamic batch padding, Sigmoid scores; evaluation threshold **0.5**; P@1, P@3, P@5, nDCG@3/@5, Micro/Macro/Weighted Precision/Recall/F1, and **F1/Precision/Recall/Support/TP/FP/FN for every one of 54 labels**.

## Kaggle setup BEFORE Cell 1

1. Create a **new Kaggle Notebook** and import this `.ipynb`.
2. From **Add Input** attach:
   - existing AAPD dataset `aapd-dataset` (contains `AAPD/test.csv`),
   - existing verified code dataset `lhrm-gpu-bundle-r-author-20261015`,
   - a **new private Kaggle Dataset**, e.g. `lhrm-r-author-frozen-artifacts`, containing your downloaded `best_dev_model_only.pt` **and** `EVIDENCE.zip` (the original ZIP or its auto-extracted contents are supported).
3. Enable **Internet ON** to download only the official pinned small BERT tokenizer/config files; select **GPU T4** if available. **Do not attach `latest.pt`, `final_model_only.pt`, `hrrm_model.joblib`, or the old training Notebook**.
4. Run cells **one at a time**. **Cells 1–4 do not read `test.csv`**. Check that the smoke test passed before running Cell 5 (first Test read). One final Test evaluation only; never tune/retrain after seeing Test scores.

> **Caveat:** Names are the original raw AAPD columns. The previously observed semantic/label-definition mapping remains unresolved; test metrics assess the supplied binary columns, not separately verified scientific meanings.
> **Dev data already used for model selection**. The numbers generated here must be labeled **Test results**; never substitute previous Dev percentages.


In [ ]:
# CELL 1 — PRE-FLIGHT: artifacts and provenance. Does NOT read Test contents.
from pathlib import Path
from io import BytesIO
import os, sys, json, hashlib, zipfile, shutil, random, time
import numpy as np
import pandas as pd
import torch

RUN_ID = "AAPD_R_AUTHOR_BCE_RECOVERY_20261015_A"
EXPECTED_EPOCH = 6
EXPECTED_STEP = 40380
EXPECTED_SEL_P1 = 0.8950
EXPECTED_LABEL_ORDER_SHA = "a3d662f3920259a94ed61d83f98f21cabe7f908d98a5662ccd9adc2413cc000b"
EXPECTED_EVIDENCE_ZIP_SHA = "cf27038453034504b37f2fef565c9c0eb47dbf288531797a8a8004c20d426d0e"
EXPECTED_BUNDLE_ZIP_SHA = "e25c583e76d44873e0bdeebcaa99973db8ac4763b79023afd2787902a8e65bbf"
EXPECTED_BUNDLE_MANIFEST_SHA = "8a6ebf146293e22e3a597f992924c84dd06ffc4fe5276d9978add3dfdd9fe9ce"
EXPECTED_EVIDENCE_FILE_SHAS = {'RUN_MANIFEST.json': '75744f07631f6f5977e4ef7941c1d1b8130e24fee389501b188bf18d0a0ab984', 'CONFIG_RESOLVED.json': '72cfcbc306a33c08d3333cebf5760fef15360004207c6c29aeb694dd47712510', 'dev_partition.json': 'fdeea51e0addcd9d52cb577275bf727bb958bb5a0d1bb313bdb9b6f93c5f541b', 'epoch_metrics.csv': 'd0e6116e045252ca075e75757c5ae1bfc6381164fdd431fbb87b87b3abacc122', 'train_history.json': '5af179ee65a4d388b669bfab69650d6a9b766971e8141e7e639e98e7745992f4', 'training_summary.json': '1febc6295962f4cbb837fedf8f92c59f0e898fddeb6cc39c091b52f77bd95f76', 'per_label_metrics.csv': '4c8f89ba4746558011e81d756fd87a76ab2c9ebb451ba2eafbcc3867a98b0067', 'step300_info.json': 'bb2ee0079d3a8c2fcfe6d7245d509eef80649d9bfe9e342388893c0c7cc5aa19', 'hrrm_heldout_comparison.json': 'fadde80fb59a41c54e1e2e80862b2de7ee4299c9d9d3808b4248b2412c14e6c3', 'inference_demo.json': 'a65373c3b08242ba3cc9adfc4f364d5cc0078aaadf0272ad9b874e0810d3b02a', 'neural_dev_predictions.json': '210539710c363c348b061af9387fd0c0f3ebf3ffd43686253e36717cc78aeee1'}
BERT_REPO = "google-bert/bert-base-uncased"
BERT_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
THRESHOLD = 0.5
MAX_LEN = 512
BATCH_SIZE = 8
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_DIR = Path('/kaggle/working/LHRM_R_AUTHOR_FROZEN_EPOCH6_TEST')
assert INPUT_ROOT.exists(), 'Run this Notebook in Kaggle, with Add Input datasets attached.'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def sha256_file(path, block=8*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(block), b''): h.update(chunk)
    return h.hexdigest()

def unique_path(matches, purpose):
    matches = sorted(set(Path(p) for p in matches))
    if len(matches) != 1:
        raise RuntimeError(f'{purpose}: expected 1 unambiguous match, got {len(matches)}: {matches}. Attach the original files only.')
    return matches[0]

# Checkpoint may be from a user Kaggle Dataset or a restored notebook Output.
ckpt_paths = list(INPUT_ROOT.rglob('best_dev_model_only.pt'))
working_ckpt = Path('/kaggle/working/best_dev_model_only.pt')
if not ckpt_paths and working_ckpt.exists(): ckpt_paths = [working_ckpt]
CHECKPOINT_PATH = unique_path(ckpt_paths, 'Frozen best checkpoint')
if CHECKPOINT_PATH.stat().st_size < 100_000_000:
    raise RuntimeError('Checkpoint file is unexpectedly small; expected full trained BERT + CSAM model.')
print('Selected checkpoint:', CHECKPOINT_PATH)
print('Checkpoint bytes:', CHECKPOINT_PATH.stat().st_size)

# Read evidence only, not Test. Kaggle might auto-extract input ZIP uploads.
evidence_zips = [p for p in INPUT_ROOT.rglob('EVIDENCE.zip') if sha256_file(p) == EXPECTED_EVIDENCE_ZIP_SHA]
if evidence_zips:
    EVIDENCE_SOURCE = unique_path(evidence_zips, 'EVIDENCE.zip')
    with zipfile.ZipFile(EVIDENCE_SOURCE) as z:
        EVIDENCE_BYTES = {n: z.read(n) for n in EXPECTED_EVIDENCE_FILE_SHAS}
    print('[EVIDENCE] Verified original ZIP SHA:', EXPECTED_EVIDENCE_ZIP_SHA)
else:
    evidence_roots = [p.parent for p in INPUT_ROOT.rglob('RUN_MANIFEST.json')
                      if sha256_file(p) == EXPECTED_EVIDENCE_FILE_SHAS['RUN_MANIFEST.json']]
    evidence_root = unique_path(evidence_roots, 'Auto-extracted evidence root')
    EVIDENCE_SOURCE = evidence_root
    EVIDENCE_BYTES = {n:(evidence_root/n).read_bytes() for n in EXPECTED_EVIDENCE_FILE_SHAS}
    print('[EVIDENCE] Using verified Kaggle auto-extracted evidence at:', evidence_root)
for n, expected in EXPECTED_EVIDENCE_FILE_SHAS.items():
    actual = hashlib.sha256(EVIDENCE_BYTES[n]).hexdigest()
    assert actual == expected, f'Evidence hash mismatch: {n}: {actual}'
RUN_MANIFEST = json.loads(EVIDENCE_BYTES['RUN_MANIFEST.json'])
CONFIG = json.loads(EVIDENCE_BYTES['CONFIG_RESOLVED.json'])
assert RUN_MANIFEST['run_id'] == RUN_ID
assert RUN_MANIFEST['best_epoch'] == EXPECTED_EPOCH
assert RUN_MANIFEST['best_step'] == EXPECTED_STEP
assert abs(RUN_MANIFEST['best_selection_p1'] - EXPECTED_SEL_P1) < 1e-10
assert RUN_MANIFEST['label_order_hash'] == EXPECTED_LABEL_ORDER_SHA
assert CONFIG['label_order_hash'] == EXPECTED_LABEL_ORDER_SHA
assert RUN_MANIFEST['selected_checkpoint'] == CHECKPOINT_PATH.name
label_df = pd.read_csv(BytesIO(EVIDENCE_BYTES['per_label_metrics.csv']))
assert len(label_df) == 54 and label_df['index'].tolist() == list(range(54))
LABELS = label_df['label'].tolist()
assert len(set(LABELS)) == 54
assert hashlib.sha256(json.dumps(LABELS).encode()).hexdigest() == EXPECTED_LABEL_ORDER_SHA
print('[EVIDENCE] PASS: run provenance, original label order, 54-label Dev metrics')

# Code bundle: either raw zip or Kaggle auto-extracted verified sources.
bundle_zips = [p for p in INPUT_ROOT.rglob('lhrm_gpu_bundle_r_author_20261015.zip')
               if sha256_file(p) == EXPECTED_BUNDLE_ZIP_SHA]
if bundle_zips:
    BUNDLE_SOURCE = unique_path(bundle_zips, 'Bundle ZIP')
    print('[BUNDLE] Found raw verified archive:', BUNDLE_SOURCE)
else:
    manifest_candidates = [p for p in INPUT_ROOT.rglob('BUNDLE_MANIFEST.json')
                           if sha256_file(p) == EXPECTED_BUNDLE_MANIFEST_SHA]
    BUNDLE_SOURCE = unique_path(manifest_candidates, 'Auto-extracted source bundle manifest')
    print('[BUNDLE] Found verified auto-extracted manifest:', BUNDLE_SOURCE)

# Only LOCATE Test here. The contents and even its header are not read until Cell 5.
possible_test_csvs = [p for p in INPUT_ROOT.rglob('test.csv')
                      if (p.parent/'dev.csv').exists() and (p.parent/'train.csv').exists()]
if possible_test_csvs:
    TEST_SOURCE = ('csv', unique_path(possible_test_csvs, 'AAPD test.csv'))
else:
    # Raw AAPD ZIP accepted. Opening ZIP central directory does NOT access rows.
    raw_aapd_zips = []
    for zp in INPUT_ROOT.rglob('*.zip'):
        if 'aapd' not in zp.name.lower(): continue
        with zipfile.ZipFile(zp) as z:
            if 'AAPD/test.csv' in z.namelist(): raw_aapd_zips.append(zp)
    TEST_SOURCE = ('zip', unique_path(raw_aapd_zips, 'AAPD.zip'))
print('[TEST] Located source path only (NOT OPENED):', TEST_SOURCE[1])
print('Device:', DEVICE, '| CUDA:', torch.cuda.is_available())
print('CELL 1 PASS — no training; no Test contents read. Next: CELL 2.')

In [ ]:
# CELL 2 — Verify and stage frozen R_AUTHOR code + acquire only pinned tokenizer/config.
from transformers import BertTokenizer, BertConfig, BertModel

WORKING = Path('/kaggle/working')
if BUNDLE_SOURCE.suffix.lower() == '.zip':
    assert sha256_file(BUNDLE_SOURCE) == EXPECTED_BUNDLE_ZIP_SHA
    with zipfile.ZipFile(BUNDLE_SOURCE, 'r') as z:
        raw_manifest = z.read('BUNDLE_MANIFEST.json')
        assert hashlib.sha256(raw_manifest).hexdigest() == EXPECTED_BUNDLE_MANIFEST_SHA
        bundle_manifest = json.loads(raw_manifest)
        source_files = {}
        for rel_path, expected in bundle_manifest['files'].items():
            content = z.read(rel_path)
            assert hashlib.sha256(content).hexdigest() == expected, rel_path
            source_files[rel_path] = content
else:
    assert sha256_file(BUNDLE_SOURCE) == EXPECTED_BUNDLE_MANIFEST_SHA
    bundle_manifest = json.loads(BUNDLE_SOURCE.read_text())
    source_files = {}
    for rel_path, expected in bundle_manifest['files'].items():
        content = (BUNDLE_SOURCE.parent/rel_path).read_bytes()
        assert hashlib.sha256(content).hexdigest() == expected, rel_path
        source_files[rel_path] = content
assert len(source_files) == 16, f'Unexpected bundle file count: {len(source_files)}'
for rel_path, content in source_files.items():
    dest = WORKING/rel_path
    assert dest.resolve().is_relative_to(WORKING.resolve()), 'Unsafe bundle path'
    dest.parent.mkdir(parents=True,exist_ok=True)
    dest.write_bytes(content)
if str(WORKING) not in sys.path: sys.path.insert(0, str(WORKING))
from src.lhrm_baseline.model import LHRMModel
from src.lhrm_baseline.data import MultiLabelDataset, collate_fn
from src.lhrm_baseline.train import predict_epoch_loader, save_predictions
from src.lhrm_baseline.metrics import compute_all_metrics, compute_per_label_metrics
from src.lhrm_baseline.inference import predict_abstract
print('Verified and staged',len(source_files),'R_AUTHOR bundle files.')

# NOTE: best_dev_model_only.pt CONTAINS the fully fine-tuned BERT weights.
# Do NOT download another full pre-trained BERT; the original pinned config+tokenizer suffice.
TOKENIZER_DIR = WORKING/'bert_author_pinned_tokenizer_config'
TOKENIZER_DIR.mkdir(parents=True,exist_ok=True)
required = ['config.json', 'vocab.txt']
if not all((TOKENIZER_DIR/x).exists() for x in required):
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(repo_id=BERT_REPO, revision=BERT_REVISION,
                          allow_patterns=['config.json','vocab.txt','tokenizer_config.json',
                                          'special_tokens_map.json','tokenizer.json'],
                          local_dir=str(TOKENIZER_DIR))
    except Exception as exc:
        raise RuntimeError('Need Kaggle Internet ON for pinned tokenizer/config; ' 
                           'alternatively attach the original official BERT tokenizer/config files.') from exc
assert all((TOKENIZER_DIR/x).exists() for x in required), 'Pinned tokenizer/config files missing.'
bert_config = BertConfig.from_pretrained(str(TOKENIZER_DIR))
assert bert_config.hidden_size == 768 and bert_config.vocab_size == 30522
TOKENIZER = BertTokenizer.from_pretrained(str(TOKENIZER_DIR))
assert len(TOKENIZER) == 30522 and TOKENIZER.pad_token_id == 0
print('CELL 2 PASS — official pinned tokenizer + architecture available; no Test read.')

In [ ]:
# CELL 3 — Load frozen checkpoint STRICTLY; no optimizer, no gradient, no Test read.
print('Loading best checkpoint to CPU for provenance checks...')
ckpt = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
assert isinstance(ckpt,dict) and isinstance(ckpt.get('model_state_dict'),dict)
assert ckpt.get('run_id') == RUN_ID, f"Unexpected run ID: {ckpt.get('run_id')}"
assert ckpt.get('variant') == 'R_AUTHOR'
assert ckpt.get('epoch') == EXPECTED_EPOCH
assert ckpt.get('step') == EXPECTED_STEP
assert abs(ckpt.get('selection_P@1',-1) - EXPECTED_SEL_P1) < 1e-10
assert ckpt.get('selection_subset') == '400_docs'
assert abs(ckpt.get('full_dev_P@1', -1) - 0.8760) < 1e-10
assert 'optimizer_state_dict' not in ckpt, 'Unexpected training checkpoint, use best_dev_model_only.pt'
sd = ckpt['model_state_dict']
assert tuple(sd['csam.label_emb'].shape) == (54, 768)
assert tuple(sd['csam.classifier.weight'].shape) == (54,1536)
print('Verified frozen checkpoint metadata: Epoch 6, Step 40380, Selection P@1=0.8950.')
print('Checkpoint file SHA-256:', sha256_file(CHECKPOINT_PATH))

# Build identical model architecture. The dummy label_emb is FULLY OVERWRITTEN by trained state_dict.
# This avoids expensive and unnecessary BERT label embedding re-computation.
BERT = BertModel(bert_config)
MODEL = LHRMModel(bert_model=BERT, num_labels=len(LABELS),
                  variant='R_AUTHOR', label_emb=torch.zeros(54,768))
MODEL.load_state_dict(sd, strict=True)
MODEL = MODEL.to(DEVICE).eval()
for p in MODEL.parameters(): p.requires_grad_(False)
assert all(not p.requires_grad for p in MODEL.parameters())
with torch.inference_mode():
    enc = TOKENIZER('Text classification sanity check.', return_tensors='pt',
                    padding='max_length', truncation=True, max_length=MAX_LEN)
    logits, _ = MODEL(enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE))
assert tuple(logits.shape) == (1, 54) and torch.isfinite(logits).all()
print('CELL 3 PASS — real trained weights loaded strictly, inference-only, no Test read.')

In [ ]:
# CELL 4 — PRE-TEST SMOKE: replay all 3 old inference examples from existing evidence.
# This MUST PASS before Test is opened. It does not involve Train / Dev / Test inputs.
SMOKE_TEXTS = [
    'We present a novel neural architecture for multi-label text classification using contextual similarity attention and label embeddings. Our hierarchical ranking approach learns cross-label dependencies and optimizes ranking metrics directly.',
    'Quantum phase transitions in disordered spin systems are investigated using numerical Monte Carlo simulations. We examine the low-temperature critical exponents and spatial correlation functions across the disordered phase boundary.',
    'Efficient algorithms for convex optimization in high-dimensional machine learning problems. We analyze stochastic gradient methods with adaptive step-sizes and prove linear convergence under strong convexity assumptions.'
]
old_demo = json.loads(EVIDENCE_BYTES['inference_demo.json'])
assert len(old_demo) == len(SMOKE_TEXTS) == 3
for i,(text,reference) in enumerate(zip(SMOKE_TEXTS,old_demo),1):
    with torch.inference_mode():
        out = predict_abstract(text=text,model=MODEL,tokenizer=TOKENIZER,
                               label_order=LABELS,device=DEVICE,hrrm_model=None,top_k=5)
    actual,expected = out['neural_top_k'],reference['neural_top_k']
    assert [x['label'] for x in actual] == [x['label'] for x in expected], f'Smoke {i}: top5 mismatch'
    max_abs_diff = max(abs(a['score']-e['score']) for a,e in zip(actual,expected))
    assert max_abs_diff < 0.002, f'Smoke {i}: model/tokenizer score mismatch {max_abs_diff}'
    print(f'[{i}/3] PASS | top1={actual[0]["label"]} | max score diff={max_abs_diff:.7f}')
SMOKE_PASSED = True
print('CELL 4 PASS — replay agrees with trained model evidence. TEST evaluation may now start.')

In [ ]:
# CELL 5 — TEST GATE: FIRST EVER TEST READ. Do not run until Cells 1–4 all PASS.
assert SMOKE_PASSED is True
assert not (OUTPUT_DIR/'test_summary_metrics.json').exists(), (
    'Existing Test results detected — do NOT blindly reevaluate/tune Test in the same output folder.')
print('=== FROZEN EVALUATION PROTOCOL: TEST CONTENTS OPENED NOW ===')
print('No optimizer, no gradient updates, no hyperparameter search, no threshold adjustments.')
# Do NOT change frozen model, label order, model choice, threshold or max_len from Test results.
if TEST_SOURCE[0] == 'csv':
    test_df = pd.read_csv(TEST_SOURCE[1])
else:
    from io import TextIOWrapper
    with zipfile.ZipFile(TEST_SOURCE[1]) as z:
        with z.open('AAPD/test.csv') as f:
            test_df = pd.read_csv(f)
metadata_cols = {'abstract','text','doc_id','id','ID'}
assert 'abstract' in test_df.columns, f'Authentic text column missing: {list(test_df.columns)}'
test_labels = [c for c in test_df.columns if c not in metadata_cols]
assert test_labels == LABELS, 'TEST LABEL ORDER OR NAMES DIFFER FROM FROZEN TRAIN ORDER — STOP WITHOUT EVALUATING!'
assert len(test_df)>0, 'Empty Test split'
assert test_df['abstract'].notna().all(), 'Missing abstract text in Test'
TEST_TEXTS = test_df['abstract'].astype(str).tolist()
raw_labels = test_df[LABELS]
assert not raw_labels.isna().any().any(), 'Missing Test target values'
TEST_Y = raw_labels.to_numpy(dtype=np.float32)
assert TEST_Y.shape == (len(TEST_TEXTS),54)
assert np.isfinite(TEST_Y).all() and np.isin(TEST_Y, [0.,1.]).all(), 'Non-binary Test target values'
print(f'Test opened: {len(TEST_TEXTS):,} docs x {len(LABELS)} labels.')
print('Label schema and order SHA: PASS', EXPECTED_LABEL_ORDER_SHA)
print('CELL 5 PASS — frozen protocol unchanged; next CELL 6 runs inference and scoring.')

In [ ]:
# CELL 6 — TEST INFERENCE & ALL METRICS. NO model training. Execute ONCE.
from torch.utils.data import DataLoader
print('=== Evaluating frozen Epoch-6 neural model on Test ===')
MODEL.eval()
for p in MODEL.parameters(): assert not p.requires_grad
TEST_DS = MultiLabelDataset(TEST_TEXTS, TEST_Y, TOKENIZER, max_len=MAX_LEN)
TEST_LOADER = DataLoader(TEST_DS, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=0, pin_memory=(DEVICE.type=='cuda'))
with torch.inference_mode():
    y_true_test,y_score_test,_ = predict_epoch_loader(MODEL,TEST_LOADER,DEVICE,None,None)
assert y_true_test.shape == y_score_test.shape == TEST_Y.shape
assert np.array_equal(y_true_test,TEST_Y), 'Test prediction order diverged from original CSV order'
assert np.isfinite(y_score_test).all() and ((y_score_test>=0)&(y_score_test<=1)).all()
TEST_METRICS = compute_all_metrics(y_true_test,y_score_test,k_values=[1,3,5],threshold=THRESHOLD)
TEST_PER_LABEL = compute_per_label_metrics(y_true_test,y_score_test,LABELS,threshold=THRESHOLD)
assert len(TEST_PER_LABEL)==54 and [x['label'] for x in TEST_PER_LABEL] == LABELS
assert all(x['tp']+x['fn']==x['support'] for x in TEST_PER_LABEL)
assert sum(x['support'] for x in TEST_PER_LABEL)==int(y_true_test.sum())
tp=sum(x['tp'] for x in TEST_PER_LABEL)
fp=sum(x['fp'] for x in TEST_PER_LABEL)
fn=sum(x['fn'] for x in TEST_PER_LABEL)
assert abs(TEST_METRICS['micro_f1'] - (2*tp/max(1,2*tp+fp+fn))) < 1e-12
print('\n--- INDEPENDENT TEST RESULTS (Neural R_AUTHOR, Frozen Epoch 6) ---')
for key in ['P@1','P@3','P@5','nDCG@3','nDCG@5','micro_f1','macro_f1',
            'weighted_f1','micro_precision','micro_recall','macro_precision','macro_recall']:
    print(f'{key:20s} {TEST_METRICS[key]:.6f} ({TEST_METRICS[key]*100:.2f}%)')
print('Per-label rows:',len(TEST_PER_LABEL),'| TP,FP,FN:',tp,fp,fn)
print('No Test-based tuning was performed.')
print('CELL 6 PASS. Next CELL 7 exports Test evidence + 54-label report.')

In [ ]:
# CELL 7 — SAVE FINAL TEST RESULTS, integrity hashes & downloadable evidence.
import csv
from IPython.display import display, FileLink
assert len(TEST_PER_LABEL)==54 and len(TEST_TEXTS)==len(y_true_test)
per_label_path = OUTPUT_DIR/'test_per_label_metrics_54.csv'
pd.DataFrame(TEST_PER_LABEL).to_csv(per_label_path,index=False)
summary = {
    'split': 'TEST', 'n_documents': len(TEST_TEXTS), 'n_labels':54,
    'run_id':RUN_ID, 'model_variant':'R_AUTHOR', 'evaluation_model':'NEURAL_ONLY',
    'checkpoint':'best_dev_model_only.pt', 'checkpoint_sha256':sha256_file(CHECKPOINT_PATH),
    'checkpoint_epoch':EXPECTED_EPOCH, 'checkpoint_step':EXPECTED_STEP,
    'selection_set':'DEV_MODEL_SELECTION_400', 'selection_P@1':EXPECTED_SEL_P1,
    'label_order_hash':EXPECTED_LABEL_ORDER_SHA,
    'threshold':THRESHOLD, 'max_len':MAX_LEN, 'batch_size':BATCH_SIZE,
    'no_training':True,'no_test_tuning':True,'hrrm_not_evaluated':True,
    'semantic_label_mapping':'UNRESOLVED_LABEL_MAPPING_ACCEPTED_BY_CLIENT_DECISION',
    'test_source_name':TEST_SOURCE[1].name,
    'metrics':{k:float(v) for k,v in TEST_METRICS.items()},
    'global_counts':{'tp':int(tp),'fp':int(fp),'fn':int(fn)}
}
(OUTPUT_DIR/'test_summary_metrics.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
record={
    'protocol':'FROZEN_ON_DEV_BEFORE_TEST','test_evaluated_once':True,
    'checkpoint_epoch':EXPECTED_EPOCH,'checkpoint_step':EXPECTED_STEP,
    'bundle_sha256':EXPECTED_BUNDLE_ZIP_SHA,
    'original_evidence_sha256':EXPECTED_EVIDENCE_ZIP_SHA,
    'checkpoint_sha256':summary['checkpoint_sha256'],
    'label_order_sha256':EXPECTED_LABEL_ORDER_SHA,
    'pinned_bert_repo':BERT_REPO,'pinned_bert_commit':BERT_REVISION,
    'threshold':THRESHOLD,'metrics_source':'src.lhrm_baseline.metrics',
    'note':'Evaluated original raw AAPD label columns, without verified external semantic mapping.'
}
(OUTPUT_DIR/'test_frozen_protocol_manifest.json').write_text(json.dumps(record,indent=2),encoding='utf-8')
save_predictions(y_true_test,y_score_test,LABELS,str(OUTPUT_DIR/'test_neural_predictions.json'))
readme = """# AAPD Test-Only Neural Evaluation — Frozen R_AUTHOR Epoch 6

This folder contains evaluation evidence from the **unchanged** best selected neural model.
Training: complete (Train 53,840 docs / 15 epochs), Model selection: Dev 400 rows.
TEST: first independent evaluation; no training, threshold tuning or checkpoint selection on Test.

Files:
- test_per_label_metrics_54.csv: **54 rows** in frozen label order, precision, recall, f1, support, tp, fp, fn (threshold 0.5)
- test_summary_metrics.json: P@1, P@3, P@5, nDCG@3/5, global Micro/Macro/Weighted PRF
- test_neural_predictions.json: row-aligned Test labels and neural scores
- test_frozen_protocol_manifest.json: provenance of model, bundle and frozen evaluation

Interpretation: per-label F1 is NOT 54 independent global Micro-F1 numbers. Raw AAPD label-name semantic mapping unresolved.
The best `.pt` weights are **NOT** included in ZIP; archive them separately.
"""
(OUTPUT_DIR/'README_TEST_REPORT.md').write_text(readme,encoding='utf-8')
files=[p for p in OUTPUT_DIR.iterdir() if p.is_file() and p.name!='SHA256SUMS.txt']
hash_lines=[f'{sha256_file(p)}  {p.name}' for p in sorted(files)]
(OUTPUT_DIR/'SHA256SUMS.txt').write_text('\n'.join(hash_lines)+'\n',encoding='utf-8')
archive = Path('/kaggle/working/LHRM_R_AUTHOR_EPOCH6_TEST_EVIDENCE.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUTPUT_DIR.iterdir()):
        if p.is_file(): z.write(p,arcname=p.name)
print('\n=== TEST ARTIFACTS ===')
print('TEST Evidence ZIP:',archive,'bytes:',archive.stat().st_size)
print('TEST ZIP SHA-256:',sha256_file(archive))
print('Per-label CSV contains',len(TEST_PER_LABEL),'rows. Global Micro-F1:',TEST_METRICS['micro_f1'])
# Stage convenience links at Kaggle's /kaggle/working root (like original Cell 8).
for path in [per_label_path, OUTPUT_DIR/'test_summary_metrics.json']:
    shutil.copy2(path, Path('/kaggle/working')/path.name)
for basename in [archive.name, per_label_path.name, 'test_summary_metrics.json']:
    display(FileLink(basename))
print('SUCCESS. Download these + retain the pre-existing best_dev_model_only.pt separately.')